In [1]:
import pandas as pd

In [ ]:
from urllib.parse import urlparse
import json


def parse_referer(url):
    p = urlparse(url)
    domain = p.netloc or None
    path = p.path if p.path not in ("", "/") else None
    return {"domain": domain, "path": path}


def parse_user_agent(series):
    if pd.isna(series):
        return {}
    return json.loads(series.replace("'", '"'))


def agg_mode(series):
    try:
        return series.mode().iloc[0]
    except (IndexError, KeyError, ValueError):
        return None

In [3]:
df_train = pd.read_csv("data/train.csv", sep=";")
df_train_labels = pd.read_csv("data/train_labels.csv", sep=";")
df_test = pd.read_csv("data/test.csv", sep=";")
df_test_users = pd.read_csv("data/test_users.csv")
df_geo_info = pd.read_csv("data/geo_info.csv", sep=";")
df_referer_vector = pd.read_csv("data/referer_vectors.csv", sep=";")

In [4]:
df_train = df_train.merge(df_train_labels, on="user_id", how="inner")
df_test = df_test_users.merge(df_test, on="user_id", how="inner")

In [5]:
df_train = df_train.merge(df_referer_vector, on="referer", how="left")
df_test = df_test.merge(df_referer_vector, on="referer", how="left")

In [6]:
df_train.head()

,request_ts,user_id,referer,geo_id,user_agent,target,component0,component1,component2,component3,component4,component5,component6,component7,component8,component9
0,1701011363,fb858e8e0a2bec074450eaf94b627fd3,https://9b48ee5/,4799,"{'browser': 'Chrome Mobile', 'browser_version'...",0,11731,4045,22213,-1184,-8992,9381,-3496,-3120,-899,16817
1,1700986581,46a5f128fd569c764a92c2eaa788095e,https://9b48ee5/,8257,"{'browser': 'Chrome Mobile', 'browser_version'...",0,11731,4045,22213,-1184,-8992,9381,-3496,-3120,-899,16817
2,1701011071,5a74e9ac53ffb21a20cce117c0ad77ba,https://9634fd0/1409e548,3150,"{'browser': 'Yandex Browser', 'browser_version...",0,12498,2451,10304,-6380,11608,3106,-2188,10573,3347,21870
3,1700992803,af735816ca19115431ae3d89518c8c91,https://9b48ee5/,2740,"{'browser': 'Chrome Mobile', 'browser_version'...",0,11731,4045,22213,-1184,-8992,9381,-3496,-3120,-899,16817
4,1701021666,364f0ae0a3f29a685c4fb5bae6033b9a,https://9b48ee5/,4863,"{'browser': 'Yandex Browser', 'browser_version...",0,11731,4045,22213,-1184,-8992,9381,-3496,-3120,-899,16817


In [7]:
df_train_referer = df_train["referer"].apply(parse_referer).apply(pd.Series)
df_test_referer = df_test["referer"].apply(parse_referer).apply(pd.Series)

In [8]:
df_train = pd.concat([df_train.drop(columns=["referer"]), df_train_referer], axis=1)
df_test = pd.concat([df_test.drop(columns=["referer"]), df_test_referer], axis=1)

In [9]:
df_train.head()

,request_ts,user_id,geo_id,user_agent,target,component0,component1,component2,component3,component4,component5,component6,component7,component8,component9,domain,path
0,1701011363,fb858e8e0a2bec074450eaf94b627fd3,4799,"{'browser': 'Chrome Mobile', 'browser_version'...",0,11731,4045,22213,-1184,-8992,9381,-3496,-3120,-899,16817,9b48ee5,None
1,1700986581,46a5f128fd569c764a92c2eaa788095e,8257,"{'browser': 'Chrome Mobile', 'browser_version'...",0,11731,4045,22213,-1184,-8992,9381,-3496,-3120,-899,16817,9b48ee5,None
2,1701011071,5a74e9ac53ffb21a20cce117c0ad77ba,3150,"{'browser': 'Yandex Browser', 'browser_version...",0,12498,2451,10304,-6380,11608,3106,-2188,10573,3347,21870,9634fd0,/1409e548
3,1700992803,af735816ca19115431ae3d89518c8c91,2740,"{'browser': 'Chrome Mobile', 'browser_version'...",0,11731,4045,22213,-1184,-8992,9381,-3496,-3120,-899,16817,9b48ee5,None
4,1701021666,364f0ae0a3f29a685c4fb5bae6033b9a,4863,"{'browser': 'Yandex Browser', 'browser_version...",0,11731,4045,22213,-1184,-8992,9381,-3496,-3120,-899,16817,9b48ee5,None


In [10]:
df_user_agent_train = df_train["user_agent"].fillna("{}").apply(parse_user_agent)
df_user_agent_test = df_test["user_agent"].fillna("{}").apply(parse_user_agent)

In [11]:
df_train = pd.concat(
    [df_train.drop(columns=["user_agent"]), df_user_agent_train.apply(pd.Series)],
    axis=1,
)
df_test = pd.concat(
    [df_test.drop(columns=["user_agent"]), df_user_agent_test.apply(pd.Series)], axis=1
)

In [12]:
df_train = df_train.merge(df_geo_info, on="geo_id", how="left")
df_test = df_test.merge(df_geo_info, on="geo_id", how="left")

In [13]:
df_train["event_count"] = 1
df_test["event_count"] = 1

In [14]:
df_train_nunique = (
    df_train.groupby("user_id")
    .agg(
        {"browser": "nunique", "os": "nunique", "domain": "nunique", "path": "nunique"}
    )
    .rename(
        columns={
            "browser": "n_browsers",
            "os": "n_os",
            "domain": "n_domains",
            "path": "n_paths",
        }
    )
)

df_test_nunique = (
    df_test.groupby("user_id")
    .agg(
        {"browser": "nunique", "os": "nunique", "domain": "nunique", "path": "nunique"}
    )
    .rename(
        columns={
            "browser": "n_browsers",
            "os": "n_os",
            "domain": "n_domains",
            "path": "n_paths",
        }
    )
    .reset_index()
)

In [15]:
component_columns = [f"component{i}" for i in range(10)]
categorical_columns = [
    "browser",
    "browser_version",
    "os",
    "os_version",
    "country_id",
    "region_id",
    "timezone_id",
    "domain",
    "path",
]

agg_dict = {}

for c in component_columns:
    agg_dict[c] = "mean"

agg_dict["event_count"] = "sum"

for c in categorical_columns:
    agg_dict[c] = agg_mode

In [16]:
agg_dict

{'component0': 'mean',
 'component1': 'mean',
 'component2': 'mean',
 'component3': 'mean',
 'component4': 'mean',
 'component5': 'mean',
 'component6': 'mean',
 'component7': 'mean',
 'component8': 'mean',
 'component9': 'mean',
 'event_count': 'sum',
 'browser': <function __main__.agg_mode(series)>,
 'browser_version': <function __main__.agg_mode(series)>,
 'os': <function __main__.agg_mode(series)>,
 'os_version': <function __main__.agg_mode(series)>,
 'country_id': <function __main__.agg_mode(series)>,
 'region_id': <function __main__.agg_mode(series)>,
 'timezone_id': <function __main__.agg_mode(series)>,
 'domain': <function __main__.agg_mode(series)>,
 'path': <function __main__.agg_mode(series)>}

In [17]:
df_train = df_train.groupby("user_id").agg(agg_dict).reset_index()
df_test = df_test.groupby("user_id").agg(agg_dict).reset_index()

In [18]:
df_train = df_train.merge(df_train_nunique, on="user_id", how="left")
df_test = df_test.merge(df_test_nunique, on="user_id", how="left")

In [19]:
df_train = df_train.merge(df_train_labels, on="user_id", how="left")

In [20]:
df_train.fillna("NaN")

,user_id,component0,component1,component2,component3,component4,component5,component6,component7,component8,...,country_id,region_id,timezone_id,domain,path,n_browsers,n_os,n_domains,n_paths,target
0,00006a10d9ff5f9d312029ac4d296430,10214.0,-1254.0,10495.0,7705.0,9547.0,5801.0,864.0,8837.0,5312.0,...,c31b4e,470e75,f6155e,6a81948,NaN,1,1,1,0,1
1,00008d5fd6bff19884138510b077a22b,10214.0,-1254.0,10495.0,7705.0,9547.0,5801.0,864.0,8837.0,5312.0,...,c31b4e,1fbfa5,e56e80,6a81948,NaN,1,1,1,0,1
2,000098f8aa8b68a125148caff0a02827,15633.0,22503.0,-2155.0,-557.0,2232.0,16084.0,-87.0,-2820.0,6578.0,...,e48e27,NaN,f0c3df,b53adc6,NaN,1,1,1,0,0
3,0000b0ec7cb23e402cc5bb0487deef4c,12055.5,10011.5,10496.0,-1075.0,9866.5,8561.5,-249.0,-7344.5,12256.5,...,c31b4e,97023a,e56e80,b56ea20,/12db98f2,2,2,2,1,1
4,0000be6a07f2fcfd68b8486d309e3a22,-3578.0,2164.0,-12993.0,5330.0,5018.0,8603.0,-2646.0,8588.0,-30424.0,...,c31b4e,380ef0,e56e80,9f1218f,NaN,1,1,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499995,fffec77a1805ac2833dc4d1d0b13aade,6616.0,-5747.0,-8118.0,2644.0,25259.0,-1288.0,2892.0,-1206.0,16894.0,...,11f5678,NaN,a3f791,9ac1e4f,NaN,1,1,1,0,0
499996,ffff1418e9af7bc4f31cb25f8c559f34,11632.0,13288.0,-4190.0,-12485.0,3084.0,14939.0,12349.0,3523.0,13239.0,...,c31b4e,97023a,e56e80,9b08d64,/12296b93,1,1,1,1,1
499997,ffffc3f4d7fe28c39ec794571ac47694,5611.0,10519.0,11801.0,9092.0,1522.0,4548.0,15459.0,6059.0,9347.0,...,c31b4e,8ccc01,e56e80,72879b4,/13e7ff6a,1,1,1,1,1
499998,ffffcc389b0db46598e56a268eb9d596,12643.0,1469.0,8146.0,3698.5,14139.0,2301.0,-3169.0,-7481.5,8780.5,...,c31b4e,8ccc01,e56e80,6972223,NaN,1,1,2,0,0


In [21]:
df_test.fillna("NaN")

,user_id,component0,component1,component2,component3,component4,component5,component6,component7,component8,...,os_version,country_id,region_id,timezone_id,domain,path,n_browsers,n_os,n_domains,n_paths
0,000098f8aa8b68a125148caff0a02827,15633.0,22503.0,-2155.0,-557.0,2232.0,16084.0,-87.0,-2820.0,6578.0,...,10,e48e27,NaN,f0c3df,b53adc6,NaN,1,1,1,0
1,0000eae7d06e8c6033731d1c1ba0c382,1489.0,15548.0,1283.0,5662.0,5217.0,11942.0,18743.0,12438.0,-2706.0,...,11,c31b4e,37246e,b4a275,72879b4,/15fd854f,1,1,1,1
2,0000f7fa001a8d49b8e83b91a1215e17,18530.0,14117.0,9243.0,-3607.0,-1048.0,5971.0,-5180.0,-5011.0,13938.0,...,10,9bd92f,NaN,100cd5d,b6630fd,NaN,1,1,1,0
3,0001c39409954fe4b4148a23154fa905,15290.0,-740.0,12145.0,11199.0,6168.0,5630.0,13574.0,10086.0,917.0,...,10,c31b4e,380ef0,e56e80,635e50c,/176eef88,1,1,1,1
4,00029eeaef3671876a5d119acff7e792,19051.0,9021.0,9045.0,-713.0,7571.0,10582.0,4131.0,14754.0,-2675.0,...,10,c31b4e,f66ff,f6155e,7f91a1d,/1283abdf,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84995,fffbf68a27011fbe76a7a856649c888f,19153.0,13108.0,8609.0,-1186.0,1192.0,7662.0,-4033.0,-4195.0,13164.0,...,10,11f5678,NaN,a3f791,b6630fd,/159c12f7,1,1,1,1
84996,fffbf8879e9a31b944237fc24895aead,11243.0,8618.0,10020.0,9558.0,23211.0,2079.0,-6186.0,1789.0,4660.0,...,10,c31b4e,470e75,f6155e,8807153,NaN,1,1,1,0
84997,fffc4301db1d1760389be359a13bc740,16429.0,13855.0,6436.0,4169.0,8693.0,15021.0,5943.0,5502.0,9233.0,...,13,b98648,241b43,ad8d84,8d7355d,/13756204,1,1,1,1
84998,fffd7f18a1234ea36ad7ceeeb50cf7ab,11243.0,8618.0,10020.0,9558.0,23211.0,2079.0,-6186.0,1789.0,4660.0,...,10,c31b4e,1fbfa5,e56e80,8807153,NaN,1,1,1,0


In [22]:
df_train.head()

,user_id,component0,component1,component2,component3,component4,component5,component6,component7,component8,...,country_id,region_id,timezone_id,domain,path,n_browsers,n_os,n_domains,n_paths,target
0,00006a10d9ff5f9d312029ac4d296430,10214.0,-1254.0,10495.0,7705.0,9547.0,5801.0,864.0,8837.0,5312.0,...,c31b4e,470e75,f6155e,6a81948,None,1,1,1,0,1
1,00008d5fd6bff19884138510b077a22b,10214.0,-1254.0,10495.0,7705.0,9547.0,5801.0,864.0,8837.0,5312.0,...,c31b4e,1fbfa5,e56e80,6a81948,None,1,1,1,0,1
2,000098f8aa8b68a125148caff0a02827,15633.0,22503.0,-2155.0,-557.0,2232.0,16084.0,-87.0,-2820.0,6578.0,...,e48e27,None,f0c3df,b53adc6,None,1,1,1,0,0
3,0000b0ec7cb23e402cc5bb0487deef4c,12055.5,10011.5,10496.0,-1075.0,9866.5,8561.5,-249.0,-7344.5,12256.5,...,c31b4e,97023a,e56e80,b56ea20,/12db98f2,2,2,2,1,1
4,0000be6a07f2fcfd68b8486d309e3a22,-3578.0,2164.0,-12993.0,5330.0,5018.0,8603.0,-2646.0,8588.0,-30424.0,...,c31b4e,380ef0,e56e80,9f1218f,None,1,1,1,0,1


In [23]:
df_test.head()

,user_id,component0,component1,component2,component3,component4,component5,component6,component7,component8,...,os_version,country_id,region_id,timezone_id,domain,path,n_browsers,n_os,n_domains,n_paths
0,000098f8aa8b68a125148caff0a02827,15633.0,22503.0,-2155.0,-557.0,2232.0,16084.0,-87.0,-2820.0,6578.0,...,10,e48e27,None,f0c3df,b53adc6,None,1,1,1,0
1,0000eae7d06e8c6033731d1c1ba0c382,1489.0,15548.0,1283.0,5662.0,5217.0,11942.0,18743.0,12438.0,-2706.0,...,11,c31b4e,37246e,b4a275,72879b4,/15fd854f,1,1,1,1
2,0000f7fa001a8d49b8e83b91a1215e17,18530.0,14117.0,9243.0,-3607.0,-1048.0,5971.0,-5180.0,-5011.0,13938.0,...,10,9bd92f,None,100cd5d,b6630fd,None,1,1,1,0
3,0001c39409954fe4b4148a23154fa905,15290.0,-740.0,12145.0,11199.0,6168.0,5630.0,13574.0,10086.0,917.0,...,10,c31b4e,380ef0,e56e80,635e50c,/176eef88,1,1,1,1
4,00029eeaef3671876a5d119acff7e792,19051.0,9021.0,9045.0,-713.0,7571.0,10582.0,4131.0,14754.0,-2675.0,...,10,c31b4e,f66ff,f6155e,7f91a1d,/1283abdf,1,1,1,1


In [24]:
from pathlib import Path

Path("preprocessed").mkdir(exist_ok=True)
df_train.to_csv("preprocessed/train.csv", index=False)
df_test.to_csv("preprocessed/test.csv", index=False)